In [ ]:
import numpy as np # only math engine
from sklearn.model_selection import train_test_split # quick & reliable data splitting
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer  # only for simple vectorization easiest way to turn text → fixed-size vectors (bag-of-words) without writing your own tokenizer from zero
import re  # for regex-based feature help later
import matplotlib.pyplot as plt  # you'll need this eventually for loss plots
import pandas as pd  # data manipulation and exporting to csv/excel
from tqdm import tqdm
import time
from datasets import load_dataset # for importing HuggingFace datasets
from google.colab import userdata # for using HuggingFace token
userdata.get('HF_TOKEN')
import json #for saving jsonl files
from pathlib import Path
import os
from google.colab import drive
drive.mount('/content/drive')

# Optional: set random seed for reproducibility
randseed = np.random.seed(42)

print("NumPy version:", np.__version__)
print("Setup complete – ready to build data and model.")
print("NP random seed: ", np.random.seed(42))

Mounted at /content/drive
NumPy version: 2.0.2
Setup complete – ready to build data and model.
NP random seed:  None


In [ ]:
%cd "drive/MyDrive/Colab Notebooks/MIS 769/Project"

/content/drive/MyDrive/Colab Notebooks/MIS 769/Project


In [ ]:
# Load your Excel file (adjust sheet name if needed)
df = pd.read_excel('Training_Set.xlsx', sheet_name='Training_Set')

# Use the final masked/original text and the binary label
# Assuming your final text column is called "Masked_Text" or similar
# and binary label is "Binary_Label"

X_raw = df['Masked_Text'].astype(str).values      # or whatever your final text column is named
y = df['Binary_Label'].values

print(f"Loaded {len(df):,} rows")
print(f"Class balance: {np.mean(y):.1%} positive (has PII)")
print(f"X_raw shape: {X_raw.shape}, y shape: {y.shape}")

Loaded 45,000 rows
Class balance: 50.4% positive (has PII)
X_raw shape: (45000,), y shape: (45000,)


In [ ]:
# ====================== LOAD GLOVE EMBEDDINGS ======================
print("Loading GloVe embeddings... (this may take 30-60 seconds)")

embeddings_index = {}
with open('glove.6B.300d.txt', encoding='utf-8') as f:
    for line in tqdm(f, total=400000):
        values = line.split()
        word = values[0]
        coefs = np.asarray(values[1:], dtype='float32')
        embeddings_index[word] = coefs

print(f"Loaded {len(embeddings_index):,} word vectors.")

Loading GloVe embeddings... (this may take 30-60 seconds)


100%|██████████| 400000/400000 [00:55<00:00, 7165.78it/s] 

Loaded 400,000 word vectors.


In [ ]:
print("Creating average embeddings for 45,000 documents...")

def text_to_embedding(text, embeddings_index, dim=300):
    words = text.lower().split()
    valid_vectors = [embeddings_index[word] for word in words if word in embeddings_index]
    if len(valid_vectors) == 0:
        return np.zeros(dim, dtype=np.float32)
    return np.mean(valid_vectors, axis=0)

X_emb = np.zeros((len(X_raw), 300), dtype=np.float32)

for i in tqdm(range(len(X_raw))):
    X_emb[i] = text_to_embedding(X_raw[i], embeddings_index)

print(f"\n✅ Done! X_emb shape: {X_emb.shape}")

Creating average embeddings for 45,000 documents...


100%|██████████| 45000/45000 [00:11<00:00, 4062.32it/s]


✅ Done! X_emb shape: (45000, 300)


In [ ]:
class ScratchNet:
    def __init__(self, input_size=300, hidden_sizes=[256, 128], dropout=0.5):
        self.layers = []
        prev_size = input_size

        # Hidden layers with He initialization
        for h in hidden_sizes:
            W = np.random.randn(prev_size, h) * np.sqrt(2.0 / prev_size)
            b = np.zeros((1, h))
            self.layers.append({'W': W, 'b': b})
            prev_size = h

        # Output layer (binary)
        W_out = np.random.randn(prev_size, 1) * np.sqrt(2.0 / prev_size)
        b_out = np.zeros((1, 1))
        self.layers.append({'W': W_out, 'b': b_out})

        self.dropout = dropout
        self.training = True
        self.activations = None
        self.dropout_masks = None

    def forward(self, X):
        X = np.asarray(X, dtype=np.float32)
        if X.ndim == 1:
            X = X.reshape(1, -1)

        activations = [X]
        dropout_masks = []

        # Hidden layers
        for layer in self.layers[:-1]:
            Z = activations[-1] @ layer['W'] + layer['b']
            A = np.maximum(0, Z)  # ReLU

            if self.training and self.dropout > 0:
                mask = (np.random.rand(*A.shape) > self.dropout) / (1 - self.dropout)
                A = A * mask
                dropout_masks.append(mask)
            else:
                dropout_masks.append(None)

            activations.append(A)

        # Output logits
        logits = activations[-1] @ self.layers[-1]['W'] + self.layers[-1]['b']
        activations.append(logits)

        self.activations = activations
        self.dropout_masks = dropout_masks
        return logits

    def backward(self, logits, y):
        y = np.asarray(y, dtype=np.float64).reshape(-1, 1)
        batch_size = y.shape[0]

        sigmoid_out = 1 / (1 + np.exp(-np.clip(logits, -500, 500)))
        current_delta = sigmoid_out - y

        for i in range(len(self.layers) - 1, -1, -1):
            layer = self.layers[i]
            A_prev = self.activations[i]

            layer['dW'] = A_prev.T @ current_delta / batch_size
            layer['db'] = np.sum(current_delta, axis=0, keepdims=True) / batch_size

            if i > 0:
                current_delta = current_delta @ layer['W'].T
                current_delta *= (self.activations[i] > 0).astype(np.float32)
                if self.dropout_masks[i-1] is not None:
                    current_delta *= self.dropout_masks[i-1]

    def update_parameters(self, lr=0.001, l2_lambda=0.001):
        """Apply SGD update with optional L2 regularization (weight decay)"""
        for layer in self.layers:
            if 'dW' in layer:
                # L2 penalty: weight decay
                layer['W'] -= lr * (layer['dW'] + l2_lambda * layer['W'])
                layer['b'] -= lr * layer['db']

    def predict_proba(self, X):
        self.training = False
        logits = self.forward(X)
        proba = 1 / (1 + np.exp(-np.clip(logits, -500, 500)))
        self.training = True
        return proba.ravel()

In [ ]:
def stable_bce_loss(logits: np.ndarray, y: np.ndarray) -> float:
    """Numerically stable binary cross-entropy loss"""
    logits = np.asarray(logits, dtype=np.float64).ravel()
    y      = np.asarray(y,     dtype=np.float64).ravel()

    if logits.shape != y.shape:
        raise ValueError(f"Shape mismatch: logits {logits.shape}, y {y.shape}")

    # Stable BCE using log-sum-exp trick
    loss_terms = (
        np.maximum(logits, 0.0)
        - logits * y
        + np.log1p(np.exp(-np.abs(logits)))
    )

    return np.mean(loss_terms)

### These inputs:
model = ScratchNet(input_size=300, hidden_sizes=[512, 256], dropout=0.5)

lr = 0.0005
epochs = 60
batch_size = 128   # larger batch size since we have more data
l2_lambda = 0.001
### Yielded these results:
### Epoch 60 | Loss: 0.6804 | Acc: 0.7351 | Time: 6.1s

### These inputs:
model = ScratchNet(input_size=300, hidden_sizes=[512, 256], dropout=0.35)

lr = 0.0005
epochs = 60
batch_size = 128   # larger batch size since we have more data
l2_lambda = 0.001
### Yielded these results:
### Epoch 60 | Loss: 0.6763 | Acc: 0.7452 | Time: 6.1s

### These inputs:
model = ScratchNet(input_size=300, hidden_sizes=[512, 256], dropout=0.3)

lr = 0.00075
epochs = 60
batch_size = 128   # larger batch size since we have more data
l2_lambda = 0.001
### Yielded these results:
### Epoch 60 | Loss: 0.6552 | Acc: 0.7772 | Time: 6.4s

### These inputs:
model = ScratchNet(input_size=300, hidden_sizes=[512, 256], dropout=0.25)

lr = 0.0009
epochs = 60
batch_size = 128   # larger batch size since we have more data
l2_lambda = 0.001
### Yielded these results:
### Epoch 60 | Loss: 0.6316 | Acc: 0.7822 | Time: 6.0s

### These inputs:
model = ScratchNet(input_size=300, hidden_sizes=[512, 256], dropout=0.2)

lr = 0.001
epochs = 60
batch_size = 128   # larger batch size since we have more data
l2_lambda = 0.001
### Yielded these results:
### Epoch 60 | Loss: 0.6074 | Acc: 0.8067 | Time: 6.0s

### These inputs:
model = ScratchNet(input_size=300, hidden_sizes=[512, 256], dropout=0.15)

lr = 0.0015
epochs = 60
batch_size = 128   # larger batch size since we have more data
l2_lambda = 0.001
### Yielded these results:
### Epoch 60 | Loss: 0.4887 | Acc: 0.8465 | Time: 10.4s

### These inputs:
model = ScratchNet(input_size=300, hidden_sizes=[512, 256], dropout=0.125)

lr = 0.00175
epochs = 60
batch_size = 128   # larger batch size since we have more data
l2_lambda = 0.001
### Yielded these results:
### Epoch 60 | Loss: 0.3973 | Acc: 0.8760 | Time: 10.2s
✅ Weights saved as 'scratchnet_weights_best.npz'

In [ ]:
# ====================== TRAINING ON THE NEW MASKED DATASET ======================
model = ScratchNet(input_size=300, hidden_sizes=[512, 256], dropout=0.125)

lr = 0.00175
epochs = 60
batch_size = 128   # larger batch size since we have more data
l2_lambda = 0.001

print("Starting training on masked 50/50 dataset...\n")

for epoch in range(epochs):
    start_time = time.time()

    model.training = True
    epoch_loss = 0.0
    correct = 0

    for start in range(0, len(X_emb), batch_size):
        end = start + batch_size
        X_batch = X_emb[start:end]
        y_batch = y[start:end]

        logits = model.forward(X_batch)
        loss = stable_bce_loss(logits, y_batch)
        epoch_loss += loss * len(y_batch)

        model.backward(logits, y_batch)
        model.update_parameters(lr=lr, l2_lambda=l2_lambda)

        preds = (model.predict_proba(X_batch) >= 0.5).astype(int)
        correct += np.sum(preds == y_batch)

    epoch_loss /= len(X_emb)
    epoch_acc = correct / len(X_emb)

    epoch_time = time.time() - start_time

    if (epoch + 1) % 5 == 0 or epoch < 5:
        print(f"Epoch {epoch+1:2d} | Loss: {epoch_loss:.4f} | Acc: {epoch_acc:.4f} | Time: {epoch_time:.1f}s")
    else:
        print(f"Epoch {epoch+1:2d} | Loss: {epoch_loss:.4f} | Acc: {epoch_acc:.4f} | Time: {epoch_time:.1f}s")

# Save the trained weights
#np.savez('scratchnet_weights_best.npz', layers=model.layers)

#print("✅ Weights saved as 'scratchnet_weights_best.npz'")

Starting training on masked 50/50 dataset...

Epoch  1 | Loss: 0.6945 | Acc: 0.4993 | Time: 10.2s
Epoch  2 | Loss: 0.6930 | Acc: 0.5142 | Time: 6.1s
Epoch  3 | Loss: 0.6909 | Acc: 0.5328 | Time: 10.2s
Epoch  4 | Loss: 0.6899 | Acc: 0.5498 | Time: 7.2s
Epoch  5 | Loss: 0.6881 | Acc: 0.5702 | Time: 10.3s
Epoch  6 | Loss: 0.6866 | Acc: 0.5919 | Time: 6.0s
Epoch  7 | Loss: 0.6851 | Acc: 0.6192 | Time: 10.2s
Epoch  8 | Loss: 0.6837 | Acc: 0.6411 | Time: 6.1s
Epoch  9 | Loss: 0.6822 | Acc: 0.6703 | Time: 10.2s
Epoch 10 | Loss: 0.6808 | Acc: 0.6933 | Time: 6.0s
Epoch 11 | Loss: 0.6798 | Acc: 0.7144 | Time: 10.2s
Epoch 12 | Loss: 0.6777 | Acc: 0.7319 | Time: 6.1s
Epoch 13 | Loss: 0.6761 | Acc: 0.7427 | Time: 10.2s
Epoch 14 | Loss: 0.6743 | Acc: 0.7515 | Time: 5.9s
Epoch 15 | Loss: 0.6729 | Acc: 0.7638 | Time: 10.1s
Epoch 16 | Loss: 0.6712 | Acc: 0.7695 | Time: 6.0s
Epoch 17 | Loss: 0.6691 | Acc: 0.7763 | Time: 10.6s
Epoch 18 | Loss: 0.6673 | Acc: 0.7836 | Time: 6.3s
Epoch 19 | Loss: 0.6653 | A

In [ ]:
# Load saved weights into a new model
model = ScratchNet(input_size=300, hidden_sizes=[512, 256], dropout=0.15)

data = np.load('scratchnet_weights_best.npz', allow_pickle=True)
model.layers = data['layers'].tolist()

print("✅ Weights loaded successfully")

###Testing

###Step 1: Load test data + sanity check

In [ ]:
# Step 1: Load test data
test_df = pd.read_excel('Testing_Set.xlsx')

X_raw_test = test_df['Masked_Text'].astype(str).values
y_test = test_df['Binary_Label'].values

print(f"✅ Test set loaded: {len(test_df):,} samples")
print(f"Class balance: {np.mean(y_test):.1%} positive")
print(f"X_raw_test shape: {X_raw_test.shape}")
print(f"y_test shape: {y_test.shape}")

# Quick check for missing values
print(f"Missing Masked_Text: {test_df['Masked_Text'].isna().sum()}")
print(f"Missing Binary_Label: {test_df['Binary_Label'].isna().sum()}")

✅ Test set loaded: 30 samples
Class balance: 26.7% positive
X_raw_test shape: (30,)
y_test shape: (30,)
Missing Masked_Text: 0
Missing Binary_Label: 0


###Step 2: Create test embeddings

In [ ]:
# Step 2: Create test embeddings (exact same as training)
print("Creating embeddings for test set...")

X_test = np.zeros((len(X_raw_test), 300), dtype=np.float32)

for i in tqdm(range(len(X_raw_test))):
    X_test[i] = text_to_embedding(X_raw_test[i], embeddings_index)

print(f"✅ Test embeddings created! X_test shape: {X_test.shape}")

Creating embeddings for test set...


100%|██████████| 30/30 [00:00<00:00, 638.69it/s]

✅ Test embeddings created! X_test shape: (30, 300)


### Step 3: Load model + weights

In [ ]:
# Step 3: Load weights - let's inspect first
weights = np.load('scratchnet_weights_best.npz', allow_pickle=True)

print("Keys in the npz file:", list(weights.keys()))
print("Type of 'layers':", type(weights['layers']))
print("Shape of 'layers':", weights['layers'].shape if hasattr(weights['layers'], 'shape') else "no shape")

# Show a bit more info
if weights['layers'].ndim == 0:
    print("Content type inside:", type(weights['layers'].item()))

Keys in the npz file: ['layers']
Type of 'layers': <class 'numpy.ndarray'>
Shape of 'layers': (3,)


In [ ]:
# Step 3: Load model and weights (fixed for your save format)
model = ScratchNet(input_size=300, hidden_sizes=[512, 256], dropout=0.125)
model.training = False

# Load the weights
weights = np.load('scratchnet_weights_best.npz', allow_pickle=True)
loaded_layers = weights['layers']          # this is the (3,) object array

# Convert to normal Python list of dicts
model.layers = loaded_layers.tolist()

print("✅ Model created and weights loaded successfully")
print(f"Number of layers: {len(model.layers)}")
print(f"Layer 0 (input→hidden1) W shape: {model.layers[0]['W'].shape}")
print(f"Layer 1 (hidden1→hidden2) W shape: {model.layers[1]['W'].shape}")
print(f"Layer 2 (hidden2→output) W shape: {model.layers[2]['W'].shape}")

✅ Model created and weights loaded successfully
Number of layers: 3
Layer 0 (input→hidden1) W shape: (300, 512)
Layer 1 (hidden1→hidden2) W shape: (512, 256)
Layer 2 (hidden2→output) W shape: (256, 1)


###Step 4: Run inference and metrics

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

In [ ]:
# Step 4: Inference and evaluation

start_time = time.time()

probs = model.predict_proba(X_test)
preds = (probs >= 0.5).astype(int)

acc = np.mean(preds == y_test)

print(f"\n🎯 Test Accuracy: {acc:.4f} ({acc*100:.2f}%)")
print(f"Inference time: {time.time() - start_time:.3f} seconds\n")

print("Classification Report:")
print(classification_report(y_test, preds, digits=4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, preds))


🎯 Test Accuracy: 0.8333 (83.33%)
Inference time: 0.036 seconds

Classification Report:
              precision    recall  f1-score   support

           0     0.9048    0.8636    0.8837        22
           1     0.6667    0.7500    0.7059         8

    accuracy                         0.8333        30
   macro avg     0.7857    0.8068    0.7948        30
weighted avg     0.8413    0.8333    0.8363        30


Confusion Matrix:
[[19  3]
 [ 2  6]]


###Step 5:Save predictions to Excel

In [ ]:
# Add predictions to the test dataframe and save
test_df['Predicted_Prob'] = probs
test_df['Predicted_Label'] = preds
test_df['Correct'] = test_df['Predicted_Label'] == test_df['Binary_Label']

# Reorder columns for nicer output
cols = ['Masked_Text', 'Binary_Label', 'Predicted_Label', 'Predicted_Prob', 'Correct']
output_df = test_df[cols]

output_df.to_excel('test_predictions.xlsx', index=False)

print("✅ Predictions saved to 'test_predictions.xlsx'")
print(f"Number of correct predictions: {test_df['Correct'].sum()}/{len(test_df)}")

✅ Predictions saved to 'test_predictions.xlsx'
Number of correct predictions: 25/30


In [ ]:
# Quick look at the label column
print("True Predictions column preview:")
print(df['True Predictions'].head(10).to_string())

# Basic stats on whether a row has PII or not
has_pii = df['True Predictions'].notna() & (df['True Predictions'].astype(str).str.strip() != '')

print("\nClass balance:")
print(f"Has PII:     {has_pii.sum():,} rows ({has_pii.mean():.1%})")
print(f"No PII:      {(~has_pii).sum():,} rows ({(~has_pii).mean():.1%})")

True Predictions column preview:
0    [(2334, 2384, 'address'), (18, 29, 'company'),...
1    [(3594, 3606, 'name'), (3521, 3577, 'credit_ca...
2    [(3701, 3717, 'name'), (3821, 3845, 'url'), (3...
3    [(3456, 3469, 'name'), (5116, 5129, 'name'), (...
4    [(56, 69, 'name'), (1137, 1150, 'name'), (841,...
5    [(67, 82, 'name'), (1410, 1425, 'name'), (789,...
6    [(4, 16, 'name'), (163, 175, 'name'), (714, 76...
7    [(87, 101, 'name'), (352, 366, 'name'), (462, ...
8    [(9, 24, 'name'), (187, 249, 'credit_card'), (...
9    [(244, 254, 'name'), (379, 389, 'name'), (453,...

Class balance:
Has PII:     45,000 rows (100.0%)
No PII:      0 rows (0.0%)


In [ ]:
import ast

In [ ]:
# Count rows that contain credit_card or ssn
has_credit_card = 0
has_ssn = 0
has_either = 0

for pred in df['True Predictions']:
    if pd.isna(pred) or str(pred).strip() == '':
        continue
    try:
        # Try to parse if it's a string representation of a list
        if isinstance(pred, str):
            items = ast.literal_eval(pred)
        else:
            items = pred

        labels = [item[2] for item in items] if isinstance(items, list) else []

        if 'credit_card' in labels:
            has_credit_card += 1
        if 'ssn' in labels:
            has_ssn += 1
        if 'credit_card' in labels or 'ssn' in labels:
            has_either += 1

    except:
        # If parsing fails, do simple string check
        pred_str = str(pred).lower()
        if 'credit_card' in pred_str:
            has_credit_card += 1
        if 'ssn' in pred_str:
            has_ssn += 1
        if 'credit_card' in pred_str or 'ssn' in pred_str:
            has_either += 1

print(f"Rows with 'credit_card': {has_credit_card:,} ({has_credit_card/len(df):.1%})")
print(f"Rows with 'ssn':         {has_ssn:,} ({has_ssn/len(df):.1%})")
print(f"Rows with either:       {has_either:,} ({has_either/len(df):.1%})")
print(f"Total rows:             {len(df):,}")

Rows with 'credit_card': 41,250 (91.7%)
Rows with 'ssn':         43,750 (97.2%)
Rows with either:       43,750 (97.2%)
Total rows:             45,000


In [ ]:
ds = load_dataset("ai4privacy/pii-masking-300k")

README.md: 0.00B [00:00, ?B/s]

data/train/1english_openpii_30k.jsonl:   0%|          | 0.00/103M [00:00<?, ?B/s]

data/train/dutch_openpii_28k.jsonl:   0%|          | 0.00/102M [00:00<?, ?B/s]

data/train/french_openpii_31k.jsonl:   0%|          | 0.00/114M [00:00<?, ?B/s]

data/train/german_openpii_30k.jsonl:   0%|          | 0.00/108M [00:00<?, ?B/s]

...

data/validation/german_openpii_8k.jsonl:   0%|          | 0.00/29.2M [00:00<?, ?B/s]

data/validation/italian_openpiii_8k.json(…):   0%|          | 0.00/28.3M [00:00<?, ?B/s]

data/validation/spanish_openpii_8k.jsonl:   0%|          | 0.00/27.7M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/177677 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/47728 [00:00<?, ? examples/s]

In [ ]:
# Filter to only English
english_ds = ds.filter(lambda x: x["language"] == "English")

print(f"Total English samples: {len(english_ds['train'])}")

Filter:   0%|          | 0/177677 [00:00<?, ? examples/s]

Filter:   0%|          | 0/47728 [00:00<?, ? examples/s]

Total English samples: 29908


In [ ]:
# Count how many have PII vs no PII
def has_pii(example):
    return len(example['privacy_mask']) > 0 if isinstance(example['privacy_mask'], list) else False

train_ds = english_ds['train']

num_pii = sum(1 for ex in train_ds if has_pii(ex))
num_no_pii = len(train_ds) - num_pii

print(f"Total English samples: {len(train_ds)}")
print(f"Has PII:     {num_pii:,}  ({num_pii/len(train_ds):.1%})")
print(f"No PII:      {num_no_pii:,}  ({num_no_pii/len(train_ds):.1%})")

Total English samples: 29908
Has PII:     26,426  (88.4%)
No PII:      3,482  (11.6%)


In [ ]:
from collections import Counter

def count_pii_types(example):
    if not isinstance(example['privacy_mask'], list):
        return []
    return [item['label'] for item in example['privacy_mask']]

# Count across all English samples
all_pii_labels = []
for example in english_ds['train']:
    all_pii_labels.extend(count_pii_types(example))

label_counts = Counter(all_pii_labels)

print("PII Type Breakdown (English subset):")
print(f"Total PII instances found: {sum(label_counts.values()):,}\n")

for label, count in label_counts.most_common():
    print(f"{label:<20} : {count:,} occurrences")

PII Type Breakdown (English subset):
Total PII instances found: 193,086

TIME                 : 14,676 occurrences
USERNAME             : 11,115 occurrences
EMAIL                : 9,759 occurrences
IDCARD               : 9,609 occurrences
SOCIALNUMBER         : 9,599 occurrences
LASTNAME1            : 9,066 occurrences
PASSPORT             : 9,020 occurrences
DRIVERLICENSE        : 8,698 occurrences
BOD                  : 8,461 occurrences
IP                   : 8,145 occurrences
GIVENNAME1           : 7,895 occurrences
CITY                 : 7,657 occurrences
STATE                : 7,590 occurrences
TITLE                : 7,560 occurrences
SEX                  : 7,496 occurrences
POSTCODE             : 7,458 occurrences
BUILDING             : 7,383 occurrences
STREET               : 7,359 occurrences
TEL                  : 7,312 occurrences
DATE                 : 6,780 occurrences
COUNTRY              : 5,875 occurrences
PASS                 : 5,637 occurrences
SECADDRESS           : 

In [ ]:
# Adjust the path if needed
with open('train.json', 'r') as f:
    data = json.load(f)

print(f"Total documents: {len(data)}")

# Quick balance check
has_pii_count = sum(1 for doc in data if any(label != "O" for label in doc['labels']))
no_pii_count = len(data) - has_pii_count

print(f"Has PII: {has_pii_count}")
print(f"No PII:  {no_pii_count}")
print(f"Balance: {has_pii_count/len(data):.1%} PII vs {no_pii_count/len(data):.1%} No PII")

Total documents: 6807
Has PII: 945
No PII:  5862
Balance: 13.9% PII vs 86.1% No PII


In [ ]:
# Count frequency of each PII label type in the full dataset
label_counter = Counter()

for doc in data:                    # 'data' is the list you loaded from train.json
    for label in doc['labels']:
        if label != 'O':
            label_counter[label] += 1

print("Kaggle PII Label Breakdown:\n")
print(f"Total non-'O' labels: {sum(label_counter.values()):,}\n")

for label, count in label_counter.most_common():
    print(f"{label:<25} : {count:,} occurrences")

Kaggle PII Label Breakdown:

Total non-'O' labels: 2,739

B-NAME_STUDENT            : 1,365 occurrences
I-NAME_STUDENT            : 1,096 occurrences
B-URL_PERSONAL            : 110 occurrences
B-ID_NUM                  : 78 occurrences
B-EMAIL                   : 39 occurrences
I-STREET_ADDRESS          : 20 occurrences
I-PHONE_NUM               : 15 occurrences
B-USERNAME                : 6 occurrences
B-PHONE_NUM               : 6 occurrences
B-STREET_ADDRESS          : 2 occurrences
I-URL_PERSONAL            : 1 occurrences
I-ID_NUM                  : 1 occurrences


In [ ]:
id_num_count = 0

for doc in subset_data:
    has_id_num = any(label == 'B-ID_NUM' or label == 'I-ID_NUM' for label in doc['labels'])
    if has_id_num:
        id_num_count += 1

print(f"Number of documents in our subset that contain ID_NUM: {id_num_count}")
print(f"Percentage: {id_num_count / len(subset_data):.2%}")

Number of documents in our subset that contain ID_NUM: 33
Percentage: 1.16%


In [ ]:
# Preview first 3 documents
for i in range(3):
    doc = data[i]
    print(f"\n--- Document {i} ---")
    print(f"Document ID: {doc.get('document', 'N/A')}")
    print(f"Full text (first 300 chars):")
    print(doc['full_text'][:300] + "..." if len(doc['full_text']) > 300 else doc['full_text'])
    print(f"Labels (first 50 tokens): {doc['labels'][:50]}")
    print(f"Has PII? → {'Yes' if any(l != 'O' for l in doc['labels']) else 'No'}")


--- Document 0 ---
Document ID: 7
Full text (first 300 chars):
Design Thinking for innovation reflexion-Avril 2021-Nathalie Sylla

Challenge & selection

The tool I use to help all stakeholders finding their way through the complexity of a project is the  mind map.

What exactly is a mind map? According to the definition of Buzan T. and Buzan B. (1999, Dessine-...
Labels (first 50 tokens): ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-NAME_STUDENT', 'I-NAME_STUDENT', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']
Has PII? → Yes

--- Document 1 ---
Document ID: 10
Full text (first 300 chars):
Diego Estrada

Design Thinking Assignment

Visualization Tool

Challenge & Selection

The elderly were having a hard time adapting to the changes we brought in our bank. As  a result of a poorly implemented linear solution, a more customer centric a

In [ ]:
import random

# Set seed for reproducibility
random.seed(42)

# Separate the documents into PII and No PII
pii_docs = [doc for doc in data if any(label != 'O' for label in doc['labels'])]
no_pii_docs = [doc for doc in data if all(label == 'O' for label in doc['labels'])]

print(f"Original PII documents: {len(pii_docs)}")
print(f"Original No PII documents: {len(no_pii_docs)}")

# Sample No PII documents for 1:2 ratio
sampled_no_pii = random.sample(no_pii_docs, 1890)

# Combine them
subset = pii_docs + sampled_no_pii

# Shuffle the subset
random.shuffle(subset)

print(f"\nFinal subset size: {len(subset)}")
print(f"   PII:    {len(pii_docs)}")
print(f"   No PII: {len(sampled_no_pii)}")
print(f"   Ratio:  1 : {len(sampled_no_pii)//len(pii_docs)}")

# Save to new JSON file
with open('pii_subset_1to2.json', 'w', encoding='utf-8') as f:
    json.dump(subset, f, ensure_ascii=False, indent=2)

print("\n✅ Subset saved as 'pii_subset_1to2.json'")

Original PII documents: 945
Original No PII documents: 5862

Final subset size: 2835
   PII:    945
   No PII: 1890
   Ratio:  1 : 2

✅ Subset saved as 'pii_subset_1to2.json'


In [ ]:
# Load the subset we just created
with open('pii_subset_1to2.json', 'r', encoding='utf-8') as f:
    subset_data = json.load(f)

print(f"Loaded subset with {len(subset_data)} documents\n")

# Extract X_raw (texts) and create binary y
X_raw = []
y = []

for doc in subset_data:
    text = doc['full_text']
    labels = doc['labels']

    has_pii = any(label != 'O' for label in labels)

    X_raw.append(text)
    y.append(1 if has_pii else 0)

X_raw = np.array(X_raw, dtype=object)
y = np.array(y, dtype=int)

# Final check
print(f"X_raw shape: {X_raw.shape}")
print(f"y shape:    {y.shape}")
print(f"Class balance:")
print(f"   Has PII (1): {np.sum(y==1)}  ({np.mean(y==1):.1%})")
print(f"   No PII (0):  {np.sum(y==0)}  ({np.mean(y==0):.1%})")

# Quick preview of first 2 examples
print("\nPreview of first 2 documents:")
for i in range(2):
    snippet = X_raw[i][:200] + "..." if len(X_raw[i]) > 200 else X_raw[i]
    print(f"[{y[i]}] {snippet}")

Loaded subset with 2835 documents

X_raw shape: (2835,)
y shape:    (2835,)
Class balance:
   Has PII (1): 945  (33.3%)
   No PII (0):  1890  (66.7%)

Preview of first 2 documents:
[0] Pandemic has deeply affected the advertising and media planning sectors such as other sectors.

Media habits and consumer habits changed during pandemic.

To reach consumer effectively we should try n...
[0] Reflection – learning launch

Challenge and selection

In my daily work I focus on helping my clients to increase the satisfaction of IT end-users  around the globe. Data-driven insights is our main d...


In [ ]:
from tqdm import tqdm

In [ ]:
# ====================== LOAD GLOVE EMBEDDINGS ======================
print("Loading GloVe embeddings... (this may take 30-60 seconds)")

embeddings_index = {}
with open('glove.6B.300d.txt', encoding='utf-8') as f:
    for line in tqdm(f, total=400000):
        values = line.split()
        word = values[0]
        coefs = np.asarray(values[1:], dtype='float32')
        embeddings_index[word] = coefs

print(f"Loaded {len(embeddings_index):,} word vectors.")

# ====================== CREATE AVERAGE EMBEDDINGS ======================
print("\nCreating average embeddings for all documents...")

def text_to_embedding(text, embeddings_index, dim=300):
    words = text.lower().split()                    # simple split for now
    valid_vectors = []

    for word in words:
        if word in embeddings_index:
            valid_vectors.append(embeddings_index[word])

    if len(valid_vectors) == 0:
        return np.zeros(dim, dtype=np.float32)      # fallback if no words found

    return np.mean(valid_vectors, axis=0)

# Convert all texts
X_emb = np.zeros((len(X_raw), 300), dtype=np.float32)

for i in tqdm(range(len(X_raw))):
    X_emb[i] = text_to_embedding(X_raw[i], embeddings_index)

print(f"\n✅ Done! X_emb shape: {X_emb.shape}")
print(f"   Sparsity: {(X_emb == 0).mean():.1%} (should be very low)")

# Quick sanity check
print("\nSample embedding norm (first document):", np.linalg.norm(X_emb[0]))

Loading GloVe embeddings... (this may take 30-60 seconds)


100%|██████████| 400000/400000 [00:49<00:00, 8053.03it/s] 


Loaded 400,000 word vectors.

Creating average embeddings for all documents...


100%|██████████| 2835/2835 [00:02<00:00, 1306.06it/s]


✅ Done! X_emb shape: (2835, 300)
   Sparsity: 0.0% (should be very low)

Sample embedding norm (first document): 3.4285505


In [ ]:
def extract_handcrafted_features(texts):
    features = np.zeros((len(texts), 16), dtype=np.float32)

    for i, text in enumerate(tqdm(texts, desc="Extracting hand-crafted features")):
        if not isinstance(text, str):
            continue

        # Basic stats
        words = text.split()
        num_words = len(words)
        num_chars = len(text)
        num_digits = sum(c.isdigit() for c in text)
        num_capitalized = sum(1 for word in words if word and word[0].isupper())

        # Feature 1-10
        features[i, 0] = num_capitalized                          # num capitalized words
        features[i, 1] = num_capitalized - (1 if words and words[0][0].isupper() else 0)  # capitalized excluding first word
        features[i, 2] = 1 if '@' in text else 0                  # has @
        features[i, 3] = num_digits                               # num digits
        features[i, 4] = num_chars                                # text length
        features[i, 5] = num_words                                # num words
        features[i, 6] = num_digits / (num_chars + 1)             # digit percentage
        features[i, 7] = num_capitalized / (num_words + 1)        # capitalized percentage

        # Name-like patterns
        features[i, 8] = sum(1 for w in words if len(w) > 2 and w[0].isupper() and w[1:].islower())  # potential names

        # Simple regex patterns
        features[i, 9] = 1 if re.search(r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b', text) else 0  # email
        features[i, 10] = 1 if re.search(r'\b\d{1,2}[-/]\d{1,2}[-/]\d{2,4}\b', text) else 0               # date-like
        features[i, 11] = 1 if re.search(r'\b\d{3}[-.\s]?\d{3}[-.\s]?\d{4}\b', text) else 0               # phone-like

        # Keyword presence
        keywords = ['student', 'name', 'id', 'email', 'phone', 'address', 'ssn']
        text_lower = text.lower()
        features[i, 12] = sum(1 for kw in keywords if kw in text_lower)

        # Consecutive capitals (common in names)
        features[i, 13] = sum(1 for w in words if len(w) > 1 and w.isalpha() and w[0].isupper() and w[1:].islower())

        # Has numbers with special chars
        features[i, 14] = 1 if re.search(r'\d+[/-]\d+', text) else 0

        # Average word length
        features[i, 15] = np.mean([len(w) for w in words]) if words else 0

    return features


# ====================== CREATE FINAL FEATURES ======================
print("Extracting hand-crafted features...")
X_hand = extract_handcrafted_features(X_raw)

# Concatenate GloVe embeddings + hand-crafted features
X_final = np.hstack([X_emb, X_hand])

print(f"\n✅ Final feature matrix created!")
print(f"Shape: {X_final.shape}  (300 GloVe + 16 hand-crafted = 316 features)")
print(f"Sparsity: {(X_final == 0).mean():.2%}")

Extracting hand-crafted features...


Extracting hand-crafted features: 100%|██████████| 2835/2835 [00:03<00:00, 769.38it/s]


✅ Final feature matrix created!
Shape: (2835, 316)  (300 GloVe + 16 hand-crafted = 316 features)
Sparsity: 1.71%


In [ ]:
import time

In [ ]:
# ====================== TRAINING WITH NEW FEATURES ======================
model = ScratchNet(input_size=316, hidden_sizes=[512, 256], dropout=0.5)

lr = 0.0005
epochs = 40
batch_size = 64

print("Starting training with GloVe + Hand-crafted features...\n")

for epoch in range(epochs):
    start_time = time.time()

    model.training = True
    epoch_loss = 0.0
    correct = 0

    for start in range(0, len(X_final), batch_size):
        end = start + batch_size
        X_batch = X_final[start:end]
        y_batch = y[start:end]

        logits = model.forward(X_batch)
        loss = stable_bce_loss(logits, y_batch)
        epoch_loss += loss * len(y_batch)

        model.backward(logits, y_batch)
        model.update_parameters(lr=lr, l2_lambda=0.001)   # L2 still on

        preds = (model.predict_proba(X_batch) >= 0.5).astype(int)
        correct += np.sum(preds == y_batch)

    epoch_loss /= len(X_final)
    epoch_acc = correct / len(X_final)

    epoch_time = time.time() - start_time

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:2d} | Loss: {epoch_loss:.4f} | Acc: {epoch_acc:.4f} | Time: {epoch_time:.1f}s")
    else:
        print(f"Epoch {epoch+1:2d} | Loss: {epoch_loss:.4f} | Acc: {epoch_acc:.4f} | Time: {epoch_time:.1f}s")

Starting training with GloVe + Hand-crafted features...

Epoch  1 | Loss: 72.6837 | Acc: 0.4868 | Time: 1.0s
Epoch  2 | Loss: 2.9551 | Acc: 0.5055 | Time: 2.1s
Epoch  3 | Loss: 1.6120 | Acc: 0.4959 | Time: 2.2s
Epoch  4 | Loss: 1.1224 | Acc: 0.5566 | Time: 1.4s
Epoch  5 | Loss: 0.9429 | Acc: 0.6226 | Time: 0.5s
Epoch  6 | Loss: 0.8524 | Acc: 0.5372 | Time: 0.5s
Epoch  7 | Loss: 0.8065 | Acc: 0.6660 | Time: 0.5s
Epoch  8 | Loss: 0.7732 | Acc: 0.6624 | Time: 0.5s
Epoch  9 | Loss: 0.7397 | Acc: 0.6395 | Time: 0.5s
Epoch 10 | Loss: 0.7702 | Acc: 0.6300 | Time: 0.5s
Epoch 11 | Loss: 0.7391 | Acc: 0.6526 | Time: 0.5s
Epoch 12 | Loss: 0.7304 | Acc: 0.6578 | Time: 0.5s
Epoch 13 | Loss: 0.7386 | Acc: 0.6279 | Time: 0.5s
Epoch 14 | Loss: 0.7160 | Acc: 0.6310 | Time: 0.5s
Epoch 15 | Loss: 0.6984 | Acc: 0.6303 | Time: 0.5s
Epoch 16 | Loss: 0.7053 | Acc: 0.6483 | Time: 0.5s
Epoch 17 | Loss: 0.7013 | Acc: 0.6681 | Time: 0.5s
Epoch 18 | Loss: 0.7093 | Acc: 0.6564 | Time: 0.5s
Epoch 19 | Loss: 0.7182 